In [1]:
import transformers
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from pyfaidx import Fasta
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
model_name = "zhihan1996/DNABERT-2-117M"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True
)

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Explicitly passing a `revision` is encouraged when loading a configuration with custom code to ensure no malicious code has been contributed in a newer revision.
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.
C:\Users\admin/.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at zhihan1996/DNABERT-2-11

In [3]:
class DNABERTClassifier(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(768, 2)
    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs 
        )
        cls = outputs[0][:, 0, :]
        return self.classifier(cls)

In [4]:
model = DNABERTClassifier(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["Wqkv"],  
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model = get_peft_model(model, lora_config)

In [ ]:
genome = Fasta(r"Homo_sapiens_CFTR_sequence.fa")

varients = pd.read_csv(r"cleaned_datasets/final_cftr_dataset.csv")
#varients = varients[~varients["Variant type"].isin(["Haplotype"])]

varients.head()

NameError: name 'Fasta' is not defined

In [6]:
varients["cause"].value_counts()

cause
1    1573
0     157
Name: count, dtype: int64

In [ ]:
START = 117287120

def extract_window(genome, pos, START, window=200):
    local_pos = pos - START
    
    start = local_pos - window
    end = local_pos + window
    
    seq = genome["7"][start:end].seq.upper()
    return seq

In [ ]:
def apply_mutation(seq, ref, alt, window=200):
    center = window

    # ensure strings
    ref = str(ref)
    alt = str(alt)

    # CASE 1: insertion (ref empty)
    if ref == "":
        mutated = seq[:center] + alt + seq[center:]

    # CASE 2: substitution or deletion
    else:
        seq_ref = seq[center:center+len(ref)]
        
        if seq_ref != ref:
            # optional debug
            # print("REF mismatch:", seq_ref, ref)
            return None
        
        mutated = seq[:center] + alt + seq[center+len(ref):]

    return mutated

In [ ]:
def tokenize_input(tokenizer, input_seq):
    tokens = tokenizer(
        input_seq,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    
    return {
        "input_ids": tokens["input_ids"].squeeze(),
        "attention_mask": tokens["attention_mask"].squeeze()
    }

In [ ]:
def create_input(ref_seq, mut_seq):
    return ref_seq + "[SEP]" + mut_seq

In [ ]:
import torch

def process_row(row, tokenizer, genome, START, window=200):
    pos = row["pos"].astype(float).astype(int)
    ref = row["ref"]
    alt = row["alt"]
    label = int(row["cause"])
    #print(ref, alt)

    # Step 1: extract
    ref_seq = extract_window(genome, pos, START, window)
    if ref_seq is None or len(ref_seq) == 0:
        return None

    # Step 2: mutate
    mut_seq = apply_mutation(ref_seq, ref, alt, window)
    if mut_seq is None:
        return None

    # Step 3: combine
    input_seq = create_input(ref_seq, mut_seq)

    # Step 4: tokenize
    tokens = tokenize_input(tokenizer, input_seq)

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "labels": torch.tensor(label)
    }

In [ ]:
def build_dataset(df, tokenizer, genome, START):
    data = []
    
    for i in range(len(df)):
        row = df.iloc[i]
        item = process_row(row, tokenizer, genome, START)
        
        if item is not None:
            data.append(item)
    
    return data

In [ ]:
dataset = build_dataset(varients, tokenizer, genome, START)


In [ ]:
type(dataset)

list

In [ ]:
labels = [seq["labels"] for seq in dataset]

In [ ]:
n = np.arange(len(dataset))

train_idx, temp_idx = train_test_split(
    n,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

temp_labels = [labels[i] for i in temp_idx]


val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

In [ ]:
train_set = Subset(dataset, train_idx)
test_set = Subset(dataset, test_idx)
val_set = Subset(dataset, val_idx)


In [ ]:
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
test_loader = DataLoader(test_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8, shuffle=True)

In [ ]:

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): DNABERTClassifier(
      (base): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(4096, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertUnpadAttention(
                (self): BertUnpadSelfAttention(
                  (dropout): Dropout(p=0.0, inplace=False)
                  (Wqkv): Linear(
                    in_features=768, out_features=2304, bias=True
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=

In [ ]:
total_loss = 0

for epoch in range(5):
    model.train()
    train_loss = 0

    for batch_train in train_loader:
        input_ids = batch_train["input_ids"].to(device)
        attention_mask = batch_train["attention_mask"].to(device)
        labels = batch_train["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_val in val_loader:
            input_ids = batch_val["input_ids"].to(device)
            attention_mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")


Epoch 1
Train Loss: 0.3664
Val Loss:   0.2206

Epoch 2
Train Loss: 0.2330
Val Loss:   0.2162

Epoch 3
Train Loss: 0.2290
Val Loss:   0.2222

Epoch 4
Train Loss: 0.2276
Val Loss:   0.2155

Epoch 5
Train Loss: 0.2269
Val Loss:   0.2210


In [ ]:
model.eval()
val_loss = 0

y_preds = []
y_labels = []
y_prob = []

with torch.no_grad():
    for batch_val in val_loader:
        input_ids = batch_val["input_ids"].to(device)
        attention_mask = batch_val["attention_mask"].to(device)
        labels = batch_val["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        prob = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        y_preds.extend(preds.cpu().numpy())
        y_labels.extend(labels.cpu().numpy())
        y_prob.extend(prob[:, 1].cpu().numpy())
        
accuracy = accuracy_score(y_labels, y_preds)
recall = recall_score(y_labels, y_preds)
roc_auc = roc_auc_score(y_labels, y_prob)
f1 = f1_score(y_labels, y_preds)
    
print("accuracy: ", accuracy_score)
print("recall: ", recall)
print("auc: ", roc_auc)
print("f1-score: ", f1)

tn, fp, fn, tp = confusion_matrix(y_labels, y_preds).ravel()

print("\nConfusion Matrix Breakdown:")
print(f"TP (correct pathogenic): {tp}")
print(f"FP (false alarm):        {fp}")
print(f"FN (missed pathogenic):  {fn}")
print(f"TN (correct benign):     {tn}")

accuracy:  <function accuracy_score at 0x000002465E1C11B0>
recall:  1.0
auc:  0.5824011931394482
f1-score:  0.9706840390879479

Confusion Matrix Breakdown:
TP (correct pathogenic): 149
FP (false alarm):        9
FN (missed pathogenic):  0
TN (correct benign):     0


In [ ]:
type(lebels)

list

In [ ]:
n = len(labels)
pat = 0
for l in labels:
    if l == 1:
        pat += 1
ben = n - pat
print(n, pat, ben)

6 3 3


In [ ]:
varients["cause"].value_counts()

cause
1    1573
0     102
Name: count, dtype: int64

In [ ]:
weight_0 = n / (2 * ben)
weight_1 = n / (2 * pat)

class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
total_loss = 0

for epoch in range(5):
    model.train()
    train_loss = 0

    for batch_train in train_loader:
        input_ids = batch_train["input_ids"].to(device)
        attention_mask = batch_train["attention_mask"].to(device)
        labels = batch_train["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_val in val_loader:
            input_ids = batch_val["input_ids"].to(device)
            attention_mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")


Epoch 1
Train Loss: 0.2267
Val Loss:   0.2155

Epoch 2
Train Loss: 0.2245
Val Loss:   0.2266

Epoch 3
Train Loss: 0.2225
Val Loss:   0.2116

Epoch 4
Train Loss: 0.2192
Val Loss:   0.2181

Epoch 5
Train Loss: 0.2089
Val Loss:   0.2054


In [ ]:
model.eval()
val_loss = 0

y_preds = []
y_labels = []
y_prob = []

with torch.no_grad():
    for batch_val in val_loader:
        input_ids = batch_val["input_ids"].to(device)
        attention_mask = batch_val["attention_mask"].to(device)
        labels = batch_val["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        prob = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        y_preds.extend(preds.cpu().numpy())
        y_labels.extend(labels.cpu().numpy())
        y_prob.extend(prob[:, 1].cpu().numpy())
        
accuracy = accuracy_score(y_labels, y_preds)
recall = recall_score(y_labels, y_preds)
roc_auc = roc_auc_score(y_labels, y_prob)
f1 = f1_score(y_labels, y_preds)
    
print("accuracy: ", accuracy_score)
print("recall: ", recall)
print("auc: ", roc_auc)
print("f1-score: ", f1)

tn, fp, fn, tp = confusion_matrix(y_labels, y_preds).ravel()

print("\nConfusion Matrix Breakdown:")
print(f"TP (correct pathogenic): {tp}")
print(f"FP (false alarm):        {fp}")
print(f"FN (missed pathogenic):  {fn}")
print(f"TN (correct benign):     {tn}")

accuracy:  <function accuracy_score at 0x000002465E1C11B0>
recall:  1.0
auc:  0.7628635346756152
f1-score:  0.9706840390879479

Confusion Matrix Breakdown:
TP (correct pathogenic): 149
FP (false alarm):        9
FN (missed pathogenic):  0
TN (correct benign):     0


In [ ]:
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    y_preds_thresh = []

    for p in y_prob:
        if p >= t:
            y_preds_thresh.append(1)
        else:
            y_preds_thresh.append(0)

    accuracy = accuracy_score(y_labels, y_preds_thresh)
    recall = recall_score(y_labels, y_preds_thresh)
    f1 = f1_score(y_labels, y_preds_thresh)

    tn, fp, fn, tp = confusion_matrix(y_labels, y_preds_thresh).ravel()

    print(f"\nThreshold: {t}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall:   {recall:.4f}")
    print(f"F1:       {f1:.4f}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")


Threshold: 0.5
Accuracy: 0.9430
Recall:   1.0000
F1:       0.9707
TP: 149 | FP: 9 | FN: 0 | TN: 0

Threshold: 0.6
Accuracy: 0.9430
Recall:   1.0000
F1:       0.9707
TP: 149 | FP: 9 | FN: 0 | TN: 0

Threshold: 0.7
Accuracy: 0.9430
Recall:   1.0000
F1:       0.9707
TP: 149 | FP: 9 | FN: 0 | TN: 0

Threshold: 0.8
Accuracy: 0.9430
Recall:   1.0000
F1:       0.9707
TP: 149 | FP: 9 | FN: 0 | TN: 0

Threshold: 0.9
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1


In [ ]:
import torch.nn.functional as F

def focal_loss(outputs, targets, alpha=None, gamma=2.0):
    ce_loss = F.cross_entropy(outputs, targets, reduction='none')
    
    pt = torch.exp(-ce_loss)
    loss = (1 - pt) ** gamma * ce_loss

    if alpha is not None:
        alpha_t = alpha[targets]
        loss = alpha_t * loss

    return loss.mean()

In [ ]:
alpha = torch.tensor([0.75, 0.25]).to(device)

loss = focal_loss(outputs, labels, alpha=alpha, gamma=2.0)

In [ ]:
total_loss = 0

for epoch in range(5):
    model.train()
    train_loss = 0

    for batch_train in train_loader:
        input_ids = batch_train["input_ids"].to(device)
        attention_mask = batch_train["attention_mask"].to(device)
        labels = batch_train["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_val in val_loader:
            input_ids = batch_val["input_ids"].to(device)
            attention_mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")


Epoch 1
Train Loss: 0.1912
Val Loss:   0.2031

Epoch 2
Train Loss: 0.1834
Val Loss:   0.1994

Epoch 3
Train Loss: 0.1708
Val Loss:   0.1941

Epoch 4
Train Loss: 0.1703
Val Loss:   0.2172

Epoch 5
Train Loss: 0.1614
Val Loss:   0.2117


In [ ]:
model.eval()
val_loss = 0

y_preds = []
y_labels = []
y_prob = []

with torch.no_grad():
    for batch_val in val_loader:
        input_ids = batch_val["input_ids"].to(device)
        attention_mask = batch_val["attention_mask"].to(device)
        labels = batch_val["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        prob = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        y_preds.extend(preds.cpu().numpy())
        y_labels.extend(labels.cpu().numpy())
        y_prob.extend(prob[:, 1].cpu().numpy())
        
accuracy = accuracy_score(y_labels, y_preds)
recall = recall_score(y_labels, y_preds)
roc_auc = roc_auc_score(y_labels, y_prob)
f1 = f1_score(y_labels, y_preds)
    
print("accuracy: ", accuracy_score)
print("recall: ", recall)
print("auc: ", roc_auc)
print("f1-score: ", f1)

tn, fp, fn, tp = confusion_matrix(y_labels, y_preds).ravel()

print("\nConfusion Matrix Breakdown:")
print(f"TP (correct pathogenic): {tp}")
print(f"FP (false alarm):        {fp}")
print(f"FN (missed pathogenic):  {fn}")
print(f"TN (correct benign):     {tn}")

accuracy:  <function accuracy_score at 0x000002465E1C11B0>
recall:  1.0
auc:  0.7807606263982103
f1-score:  0.9738562091503268

Confusion Matrix Breakdown:
TP (correct pathogenic): 149
FP (false alarm):        8
FN (missed pathogenic):  0
TN (correct benign):     1


In [ ]:
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    y_preds_thresh = []

    for p in y_prob:
        if p >= t:
            y_preds_thresh.append(1)
        else:
            y_preds_thresh.append(0)

    accuracy = accuracy_score(y_labels, y_preds_thresh)
    recall = recall_score(y_labels, y_preds_thresh)
    f1 = f1_score(y_labels, y_preds_thresh)

    tn, fp, fn, tp = confusion_matrix(y_labels, y_preds_thresh).ravel()

    print(f"\nThreshold: {t}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall:   {recall:.4f}")
    print(f"F1:       {f1:.4f}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")


Threshold: 0.5
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.6
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.7
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.8
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.9
Accuracy: 0.9430
Recall:   0.9933
F1:       0.9705
TP: 148 | FP: 8 | FN: 1 | TN: 1


In [ ]:
alpha = torch.tensor([0.85, 0.15]).to(device)

loss = focal_loss(outputs, labels, alpha=alpha, gamma=2.0)

In [ ]:
total_loss = 0

for epoch in range(5):
    model.train()
    train_loss = 0

    for batch_train in train_loader:
        input_ids = batch_train["input_ids"].to(device)
        attention_mask = batch_train["attention_mask"].to(device)
        labels = batch_train["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_val in val_loader:
            input_ids = batch_val["input_ids"].to(device)
            attention_mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")
model.eval()
val_loss = 0

y_preds = []
y_labels = []
y_prob = []

with torch.no_grad():
    for batch_val in val_loader:
        input_ids = batch_val["input_ids"].to(device)
        attention_mask = batch_val["attention_mask"].to(device)
        labels = batch_val["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        prob = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        y_preds.extend(preds.cpu().numpy())
        y_labels.extend(labels.cpu().numpy())
        y_prob.extend(prob[:, 1].cpu().numpy())
        
accuracy = accuracy_score(y_labels, y_preds)
recall = recall_score(y_labels, y_preds)
roc_auc = roc_auc_score(y_labels, y_prob)
f1 = f1_score(y_labels, y_preds)
    
print("accuracy: ", accuracy_score)
print("recall: ", recall)
print("auc: ", roc_auc)
print("f1-score: ", f1)

tn, fp, fn, tp = confusion_matrix(y_labels, y_preds).ravel()

print("\nConfusion Matrix Breakdown:")
print(f"TP (correct pathogenic): {tp}")
print(f"FP (false alarm):        {fp}")
print(f"FN (missed pathogenic):  {fn}")
print(f"TN (correct benign):     {tn}")
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    y_preds_thresh = []

    for p in y_prob:
        if p >= t:
            y_preds_thresh.append(1)
        else:
            y_preds_thresh.append(0)

    accuracy = accuracy_score(y_labels, y_preds_thresh)
    recall = recall_score(y_labels, y_preds_thresh)
    f1 = f1_score(y_labels, y_preds_thresh)

    tn, fp, fn, tp = confusion_matrix(y_labels, y_preds_thresh).ravel()

    print(f"\nThreshold: {t}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall:   {recall:.4f}")
    print(f"F1:       {f1:.4f}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")


Epoch 1
Train Loss: 0.1582
Val Loss:   0.2111

Epoch 2
Train Loss: 0.1576
Val Loss:   0.2054

Epoch 3
Train Loss: 0.1524
Val Loss:   0.2052

Epoch 4
Train Loss: 0.1484
Val Loss:   0.2064

Epoch 5
Train Loss: 0.1476
Val Loss:   0.1949
accuracy:  <function accuracy_score at 0x000002465E1C11B0>
recall:  1.0
auc:  0.6733780760626399
f1-score:  0.9738562091503268

Confusion Matrix Breakdown:
TP (correct pathogenic): 149
FP (false alarm):        8
FN (missed pathogenic):  0
TN (correct benign):     1

Threshold: 0.5
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.6
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.7
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.8
Accuracy: 0.9430
Recall:   0.9933
F1:       0.9705
TP: 148 | FP: 8 | FN: 1 | TN: 1

Threshold: 0.9
Accuracy: 0.9114
Recall:   0.9597
F1:       0.9533
TP: 143 | FP: 8 | FN: 6 | TN: 1


In [ ]:
alpha = torch.tensor([0.90, 0.10]).to(device)

loss = focal_loss(outputs, labels, alpha=alpha, gamma=2.0)

In [ ]:
total_loss = 0

for epoch in range(5):
    model.train()
    train_loss = 0

    for batch_train in train_loader:
        input_ids = batch_train["input_ids"].to(device)
        attention_mask = batch_train["attention_mask"].to(device)
        labels = batch_train["labels"].to(device)
        
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_val in val_loader:
            input_ids = batch_val["input_ids"].to(device)
            attention_mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")
model.eval()
val_loss = 0

y_preds = []
y_labels = []
y_prob = []

with torch.no_grad():
    for batch_val in val_loader:
        input_ids = batch_val["input_ids"].to(device)
        attention_mask = batch_val["attention_mask"].to(device)
        labels = batch_val["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        prob = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        y_preds.extend(preds.cpu().numpy())
        y_labels.extend(labels.cpu().numpy())
        y_prob.extend(prob[:, 1].cpu().numpy())
        
accuracy = accuracy_score(y_labels, y_preds)
recall = recall_score(y_labels, y_preds)
roc_auc = roc_auc_score(y_labels, y_prob)
f1 = f1_score(y_labels, y_preds)
    
print("accuracy: ", accuracy_score)
print("recall: ", recall)
print("auc: ", roc_auc)
print("f1-score: ", f1)

tn, fp, fn, tp = confusion_matrix(y_labels, y_preds).ravel()

print("\nConfusion Matrix Breakdown:")
print(f"TP (correct pathogenic): {tp}")
print(f"FP (false alarm):        {fp}")
print(f"FN (missed pathogenic):  {fn}")
print(f"TN (correct benign):     {tn}")
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    y_preds_thresh = []

    for p in y_prob:
        if p >= t:
            y_preds_thresh.append(1)
        else:
            y_preds_thresh.append(0)

    accuracy = accuracy_score(y_labels, y_preds_thresh)
    recall = recall_score(y_labels, y_preds_thresh)
    f1 = f1_score(y_labels, y_preds_thresh)

    tn, fp, fn, tp = confusion_matrix(y_labels, y_preds_thresh).ravel()

    print(f"\nThreshold: {t}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall:   {recall:.4f}")
    print(f"F1:       {f1:.4f}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}")


Epoch 1
Train Loss: 0.1488
Val Loss:   0.2036

Epoch 2
Train Loss: 0.1457
Val Loss:   0.2053

Epoch 3
Train Loss: 0.1422
Val Loss:   0.2021

Epoch 4
Train Loss: 0.1366
Val Loss:   0.2039

Epoch 5
Train Loss: 0.1338
Val Loss:   0.2074
accuracy:  <function accuracy_score at 0x000002465E1C11B0>
recall:  1.0
auc:  0.662938105891126
f1-score:  0.9738562091503268

Confusion Matrix Breakdown:
TP (correct pathogenic): 149
FP (false alarm):        8
FN (missed pathogenic):  0
TN (correct benign):     1

Threshold: 0.5
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.6
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.7
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.8
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1

Threshold: 0.9
Accuracy: 0.9494
Recall:   1.0000
F1:       0.9739
TP: 149 | FP: 8 | FN: 0 | TN: 1
